<a href="https://colab.research.google.com/github/cashby-890/ST-554-HW6/blob/main/ST554_Homework_Six_Part_One_Cody_Ashby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ST 554 Homework Six Part I by Cody Ashby**

## ***More SQL Practice***

1. Let's re-establish our connection to the `lahman` database via `sqlite3`.

In [2]:
import sqlite3
con = sqlite3.connect("lahman_1871-2022.sqlite")

Next, we'll set up the schema so we can access the tables in the database.

In [3]:
cursor = con.cursor()
get_schema='''
        select *
        from sqlite_schema
        where type="table";
        '''
cursor.execute(get_schema)
result=cursor.fetchall()
cursor.close()

Lastly, we'll use `pandas` and the `read_sql()` command to return the collection of tables as a data frame.

In [4]:
import pandas as pd
lahman_df=pd.read_sql(get_schema,con)
lahman_df

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


2. Now, we'll use SQL to create a table of summary statistics for pitchers that were inducted into the Hall of Fame during the regular season.

In [21]:
inner_1="""
    select distinct P.playerID,sum(P.GS),sum(P.G),sum(P.W),sum(P.L),sum(P.IPOuts),sum(P.CG),sum(P.SHO),sum(P.SV) from Pitching as P
    inner join HallOfFame as H on H.playerID=P.playerID
    where H.inducted='Y'
    group by P.playerID
    """
HOF_Pitching_Stats=pd.read_sql(inner_1,con)
HOF_Pitching_Stats

,playerID,sum(P.GS),sum(P.G),sum(P.W),sum(P.L),sum(P.IPOuts),sum(P.CG),sum(P.SHO),sum(P.SV)
0,alexape01,599,696,373,208,15570,437,90,32
1,ansonca01,0,3,0,1,12,0,0,1
2,becklja01,1,1,0,1,12,0,0,0
3,bendech01,334,459,212,127,9051,255,40,34
4,blylebe01,685,692,287,250,14910,242,60,0
...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11
104,wrighge01,0,3,0,1,15,0,0,0
105,wrighha01,8,36,4,4,301,0,0,14
106,wynnea01,612,691,300,244,13692,290,49,15


3. We'll now look at the batting statistics for those Hall of Fame pitchers.

In [18]:
double_inner_join="""
    select distinct B.playerID,sum(B.AB),sum(B.R),sum(B.H),sum(B.HR),sum(B.RBI),sum(B.BB),sum(B.SO) from Batting as B
      inner join HallOfFame as H on H.playerID=B.playerID
      inner join Pitching as P on P.playerID=B.playerID
    where H.inducted='Y'
    group by B.playerID
    """
HOF_Pitcher_Batting_Stats=pd.read_sql(double_inner_join,con)
HOF_Pitcher_Batting_Stats

,playerID,sum(B.AB),sum(B.R),sum(B.H),sum(B.HR),sum(B.RBI),sum(B.BB),sum(B.SO)
0,alexape01,38010,3234,7938,231,3423,1617,5796
1,ansonca01,20562,3998,6870,194,4150,1968,660
2,becklja01,9551,1603,2938,87,1581,616,526
3,bendech01,18352,1632,3888,96,1856,1200,2288
4,blylebe01,10824,456,1416,0,600,120,4632
...,...,...,...,...,...,...,...,...
103,willivi01,19409,1391,3224,13,1092,1053,2587
104,wrighge01,5746,1330,1732,22,652,136,238
105,wrighha01,3252,732,896,16,452,148,56
106,wynnea01,39192,3128,8395,391,3979,3243,7590


4. To wrap things up, we'll merge these two tables together to get the pitching and batting stats for all of the Hall of Fame pitchers.

In [22]:
HOF_Pitcher_Stats=pd.merge(left=HOF_Pitching_Stats,right=HOF_Pitcher_Batting_Stats)
HOF_Pitcher_Stats

,playerID,sum(P.GS),sum(P.G),sum(P.W),sum(P.L),sum(P.IPOuts),sum(P.CG),sum(P.SHO),sum(P.SV),sum(B.AB),sum(B.R),sum(B.H),sum(B.HR),sum(B.RBI),sum(B.BB),sum(B.SO)
0,alexape01,599,696,373,208,15570,437,90,32,38010,3234,7938,231,3423,1617,5796
1,ansonca01,0,3,0,1,12,0,0,1,20562,3998,6870,194,4150,1968,660
2,becklja01,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581,616,526
3,bendech01,334,459,212,127,9051,255,40,34,18352,1632,3888,96,1856,1200,2288
4,blylebe01,685,692,287,250,14910,242,60,0,10824,456,1416,0,600,120,4632
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11,19409,1391,3224,13,1092,1053,2587
104,wrighge01,0,3,0,1,15,0,0,0,5746,1330,1732,22,652,136,238
105,wrighha01,8,36,4,4,301,0,0,14,3252,732,896,16,452,148,56
106,wynnea01,612,691,300,244,13692,290,49,15,39192,3128,8395,391,3979,3243,7590
